# OpenInsight Data Ingestion Pipeline — Kaggle Edition

Runs the full ingestion pipeline on Kaggle GPU. Parses PDFs/XML, chunks, embeds, and writes to Zilliz Cloud + MongoDB Atlas.

## API Keys

Add via **Kaggle → Add-ons → Secrets**:

**Required:**
- `ZILLIZ_URI`, `ZILLIZ_TOKEN` — vector DB
- `MONGODB_URL`, `MONGODB_DB` — document store
- `NVIDIA_NIM_API_KEY` — for embedding model downloads

**Optional:**
- `NCBI_API_KEY`, `NCBI_EMAIL` — PubMed API
- `HF_API_TOKEN` — HuggingFace (if using HF embedder)
- `COHERE_API_KEY` — Cohere (if using Cohere embedder)

## 1. Install Dependencies

Installs runtime packages + Java (GROBID) + Tesseract (OCR).

In [ ]:
%pip install -q 'numpy<2.0'
%pip install -q --force-reinstall --no-deps pandas

# Core dependencies
%pip install -q \
    fastapi uvicorn python-dotenv pydantic pydantic-settings \
    pymongo motor pypdf2 pdfplumber beautifulsoup4 requests httpx lxml biopython \
    sentence-transformers transformers torch \
    'pymilvus>=2.5,<2.6' langchain langchain-community openai redis \
    tqdm loguru tenacity python-slugify pytesseract Pillow scispacy \
    reportlab

# Verify core imports
import numpy as np
print(f'numpy: {np.__version__}')
import pandas as pd
print(f'pandas: {pd.__version__}')
from pymilvus import MilvusClient
print('pymilvus: OK')

# System packages for GROBID and OCR
!apt-get update -qq
!apt-get install -qq -y default-jre wget unzip tesseract-ocr > /dev/null 2>&1

import os, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/openinsight')
if REPO_DIR.exists():
    %cd /kaggle/working/openinsight
    !git pull origin restruct
else:
    !git clone -b restruct https://github.com/Adi103-ETAI/openinsight.git /kaggle/working/openinsight

sys.path.insert(0, str(REPO_DIR))
print('Environment ready.')

## 2. Configure Environment

Sets up secrets and env vars for the ingestion pipeline. Embedding/reranking use Kaggle GPU via `EMBED_PROVIDER=local` and `RERANK_PROVIDER=local`.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
from pathlib import Path


def get_secret(name: str, default: str = '') -> str:
    try:
        client = UserSecretsClient()
        value = client.get_secret(name)
        return default if value is None else value
    except Exception:
        return default


ZILLIZ_URI         = get_secret('ZILLIZ_URI', '')
ZILLIZ_TOKEN       = get_secret('ZILLIZ_TOKEN', '')
MONGODB_URL        = get_secret('MONGODB_URL', '')
MONGODB_DB         = get_secret('MONGODB_DB', 'openinsight')
NVIDIA_NIM_API_KEY = get_secret('NVIDIA_NIM_API_KEY', '')
HF_API_TOKEN       = get_secret('HF_API_TOKEN', '')
COHERE_API_KEY     = get_secret('COHERE_API_KEY', '')
NCBI_API_KEY       = get_secret('NCBI_API_KEY', '')
NCBI_EMAIL         = get_secret('NCBI_EMAIL', 'sentarc.ai@gmail.com')

# ── Ingestion config ──
SOURCE_TYPE = 'pubmed'
DATA_DIR = '/kaggle/input/your-dataset-name'
BATCH_SIZE = 10
RECREATE_INDEX = False
RESUME = True
RESET = False
SINGLE_FILE = None

# ── Environment variables ──
os.environ['MONGODB_URL']          = MONGODB_URL
os.environ['MONGODB_DB']           = MONGODB_DB
os.environ['VECTOR_URI']           = ZILLIZ_URI
os.environ['VECTOR_TOKEN']         = ZILLIZ_TOKEN
os.environ['MILVUS_CLOUD']         = 'true'
os.environ['VECTOR_COLLECTION_V2'] = 'openinsight_v2'
os.environ['NVIDIA_NIM_API_KEY']   = NVIDIA_NIM_API_KEY
os.environ['NVIDIA_NIM_BASE_URL']  = 'https://integrate.api.nvidia.com/v1'
os.environ['NIM_MODEL']            = 'meta/llama-3.1-70b-instruct'
os.environ['EMBED_PROVIDER']       = 'local'
os.environ['RERANK_PROVIDER']      = 'local'

if NCBI_API_KEY:
    os.environ['NCBI_API_KEY'] = NCBI_API_KEY
os.environ['NCBI_EMAIL'] = NCBI_EMAIL

if HF_API_TOKEN:
    os.environ['HF_API_TOKEN'] = HF_API_TOKEN
if COHERE_API_KEY:
    os.environ['COHERE_API_KEY'] = COHERE_API_KEY

print('Environment configured.')

## 3. Start GROBID and Verify GPU

Kaggle does not ship with GROBID. This cell starts it locally and confirms GPU availability.

In [ ]:
# Start GROBID for PDF parsing and verify GPU availability
import subprocess
import time
import httpx
import torch

GROBID_DIR = Path('/kaggle/working/grobid')
GROBID_BIN = GROBID_DIR / 'grobid-0.9.0' / 'bin' / 'grobid-server'
GROBID_URL = 'http://localhost:8070'

if not GROBID_BIN.exists():
    !wget -q https://github.com/grobidOrg/grobid/releases/download/0.9.0/grobid-0.9.0.zip -O /kaggle/working/grobid-0.9.0.zip
    !unzip -q -o /kaggle/working/grobid-0.9.0.zip -d /kaggle/working/grobid

if 'grobid_proc' not in globals() or grobid_proc.poll() is not None:
    grobid_proc = subprocess.Popen(
        [str(GROBID_BIN), '--port', '8070'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

os.environ['GROBID_URL'] = GROBID_URL

for attempt in range(5):
    try:
        response = httpx.get(f'{GROBID_URL}/api/health', timeout=10)
        if response.status_code == 200:
            print('GROBID 0.9.0: ready (health endpoint)')
            break
    except:
        pass
    try:
        response = httpx.get(f'{GROBID_URL}/api/isalive', timeout=10)
        if response.status_code == 200:
            print('GROBID: ready (isalive fallback)')
            break
    except:
        pass
    time.sleep(3)
else:
    print('Warning: GROBID not responding. Will use fallback parsers.')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('Configured input directories:')
print(f'  Repo: {REPO_DIR}')
print(f'  Data: {DATA_DIR}')

## 3.5. Wipe Old Data (First Run Only)

Run this cell **once** before your first ingestion to clear any old/bad data.

In [ ]:
# ── WIPE ALL DATA (Run this once before first ingestion) ──
from pymilvus import MilvusClient
from pymongo import MongoClient

# Wipe Zilliz Cloud collection
zilliz = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz.has_collection('openinsight_v2'):
    zilliz.drop_collection('openinsight_v2')
    print('Zilliz: Dropped collection openinsight_v2')
else:
    print('Zilliz: Collection does not exist yet')

# Wipe all MongoDB collections
mongo = MongoClient(MONGODB_URL)
db = mongo[MONGODB_DB]
for col_name in ['documents_v2', 'chunks_v2', 'failed_documents',
                  'ingestion_checkpoints', 'ingestion_metrics']:
    result = db[col_name].delete_many({})
    print(f'MongoDB {col_name}: Deleted {result.deleted_count} docs')

print('All data wiped! Ready for fresh ingestion.')

## 4. Run Ingestion

Uses the repository's native `IngestionPipeline` — handles parsing, deduplication,
enrichment, chunking, embedding, and vector indexing.

In [ ]:
# Run the ingestion pipeline using the repo's native orchestrator
import asyncio
from pathlib import Path

from src.ingestion.pipeline import IngestionPipeline
from src.ingestion.run_ingestion import SOURCES


def list_candidate_files(directory: str) -> list[Path]:
    root = Path(directory)
    if not root.exists():
        return []
    return sorted(
        [
            path
            for path in root.rglob('*')
            if path.is_file() and path.suffix.lower() in {'.pdf', '.xml'}
        ]
    )


def print_source_catalog() -> None:
    print('Available sources:')
    for source in SOURCES:
        print(f'  - {source}')


print_source_catalog()
print(f'Candidate files in DATA_DIR: {len(list_candidate_files(DATA_DIR))}')

async def run_ingestion() -> dict:
    pipeline = IngestionPipeline()

    print('Starting ingestion with these settings:')
    print(f'  source: {SOURCE_TYPE}')
    print(f'  directory: {DATA_DIR}')
    print(f'  batch_size: {BATCH_SIZE}')
    print(f'  recreate_index: {RECREATE_INDEX}')
    print(f'  resume: {RESUME}')
    print(f'  reset: {RESET}')
    print(f'  single_file: {SINGLE_FILE}')
    print('=' * 60)

    summary = await pipeline.ingest_directory(
        directory=DATA_DIR,
        source=SOURCE_TYPE,
        recreate_index=RECREATE_INDEX,
        batch_size=BATCH_SIZE,
        resume=RESUME,
        reset=RESET,
        single_file=SINGLE_FILE,
    )

    print('')
    print('=' * 60)
    print('INGESTION COMPLETE')
    print('=' * 60)
    for key, value in summary.items():
        print(f'  {key}: {value}')

    recent_runs = await pipeline.monitor.get_recent_runs(limit=5)
    print('Recent ingestion runs:')
    for run in recent_runs:
        print(f"  - {run.get('source_type', 'unknown')} | {run.get('status', 'unknown')} | {run.get('documents_stored', 0)} docs | {run.get('chunks_embedded', 0)} chunks")

    return summary

summary = await run_ingestion()

## 5. Verify Storage and Metrics

Confirm ingestion wrote to Zilliz Cloud, MongoDB, checkpoints, and run metrics.

In [ ]:
# Verify data written to Zilliz Cloud, MongoDB, checkpoints, and run metrics
from pymilvus import MilvusClient
from pymongo import MongoClient

from src.ingestion.monitoring import IngestionMonitor
from src.data.mongo.connection import get_mongo_db

collection_name = os.environ.get('VECTOR_COLLECTION_V2', 'openinsight_v2')

zilliz_client = MilvusClient(uri=ZILLIZ_URI, token=ZILLIZ_TOKEN)
if zilliz_client.has_collection(collection_name):
    stats = zilliz_client.get_collection_stats(collection_name)
    print(f'Zilliz collection: {collection_name}')
    print(f"Row count: {stats.get('row_count', 'unknown')}")
else:
    print(f'Collection not found: {collection_name}')

mongo_client = MongoClient(MONGODB_URL)
db = mongo_client[MONGODB_DB]
print(f'MongoDB database: {MONGODB_DB}')
print(f"documents_v2: {db.documents_v2.count_documents({})}")
print(f"chunks_v2: {db.chunks_v2.count_documents({})}")
print(f"failed_documents: {db.failed_documents.count_documents({})}")
print(f"ingestion_checkpoints: {db.ingestion_checkpoints.count_documents({})}")
print(f"ingestion_metrics: {db.ingestion_metrics.count_documents({})}")

mongo_db = get_mongo_db(MONGODB_DB)
monitor = IngestionMonitor(mongo_db)

async def show_monitoring() -> None:
    storage_stats = await monitor.get_storage_stats()
    print('Storage stats by source:')
    print(storage_stats)

    recent_runs = await monitor.get_recent_runs(limit=3)
    print('Latest run metrics:')
    for run in recent_runs:
        print(
            f"  - {run.get('source_type', 'unknown')} | "
            f"{run.get('status', 'unknown')} | "
            f"docs={run.get('documents_stored', 0)} | "
            f"chunks={run.get('chunks_embedded', 0)} | "
            f"duration={run.get('duration_seconds', 0)}s"
        )

await show_monitoring()

try:
    mongo_client.close()
    zilliz_client.close()
    print('Cleaned up.')
except Exception:
    pass